In [50]:
import pandas as pd
import numpy as np
import re

In [51]:
df = pd.read_csv("../dataset/job_market_clean.csv")

In [52]:
df.shape

(2873, 13)

In [53]:
df.head()

,slug,company_name,title,description,remote,url,tags,job_types,location,created_at,posted_date,posted_year,posted_month
0,software-engineer-price-comparison-portal-berl...,Preiswecker,Software Engineer,Preiswecker is a price comparison portal in Ge...,False,https://www.preiswecker.com/,Software Engineering | E-Commerce,Full-time,Berlin,2026-08-12 06:40:00,2026-08-12,2026,August
1,senior-product-manager-356253,Verve,Senior Product Manager,Who We Are Verve has created a more efficient ...,False,https://www.arbeitnow.com/jobs/companies/verve...,Product Management,NaN,Unknown,2026-10-01 05:55:15,2026-10-01,2026,October
2,working-student-launch-operations-parsdorf-bav...,isaraerospace,Working Student Launch Operations (m/f/d),Mission Brief As Working Student Launch Operat...,False,https://www.arbeitnow.com/jobs/companies/isara...,Mission & Launch Operations,NaN,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October
3,senior-mechanical-engineer-parsdorf-bavaria-38...,isaraerospace,Senior Mechanical Engineer (f/m/d),Mission Brief We deliver the half of the rocke...,False,https://www.arbeitnow.com/jobs/companies/isara...,Ground Systems,NaN,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October
4,mes-process-and-system-engineer-parsdorf-bavar...,isaraerospace,MES Process and System Engineer (m/f/d),Mission Brief&nbsp; Help build the digital man...,False,https://www.arbeitnow.com/jobs/companies/isara...,Production | Operations & Quality,NaN,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October


In [54]:
# Remote Status
df["remote_status"] = np.where(
    df["remote"], "Remote", "On-site"
)

In [55]:
df["remote_status"].value_counts()

remote_status
On-site    2662
Remote      211
Name: count, dtype: int64

In [56]:
# Job Category
title = df["title"].str.lower()

In [57]:
df["job_category"] = np.select(
    [
        title.str.contains("data|analyst|analytics", na=False),
        title.str.contains("software|developer|engineer", na=False),
        title.str.contains("marketing|sales", na=False),
        title.str.contains("finance|accounting", na=False),
        title.str.contains("hr|human resources", na=False)
    ],
    [
        "Data & Analytics",
        "Software & IT",
        "Sales & Marketing",
        "Finance",
        "HR"
    ],
    default="Other"
)

In [58]:
df["job_category"].value_counts()

job_category
Other                1678
Software & IT         635
Sales & Marketing     279
Data & Analytics      175
HR                     58
Finance                48
Name: count, dtype: int64

In [59]:
# Experience Level
df["experience_level"] = np.select(
    [
        title.str.contains("intern|internship|junior|entry", na=False),
        title.str.contains("senior|sr\\.", na=False),
        title.str.contains("lead|manager|head|director", na=False)
    ],
    [
        "Entry Level",
        "Senior",
        "Lead / Manager"
    ],
    default="Mid Level / Other"
)

In [60]:
df["experience_level"].value_counts()

experience_level
Mid Level / Other    1578
Senior                683
Lead / Manager        492
Entry Level           120
Name: count, dtype: int64

In [61]:
# Clean Location
df["city"] = (
    df["location"]
    .str.split(",")
    .str[0]
    .str.strip()
)

In [62]:
df["city"].value_counts().head(15)

city
Berlin               584
Munich               265
Hamburg              167
München              120
London                72
Unknown               67
Paris                 53
Köln                  50
Cologne               48
Remote                47
Stuttgart             45
Frankfurt am Main     45
Germany               41
Düsseldorf            39
Zürich                25
Name: count, dtype: int64

In [65]:
# Extract Skills
skills = [
    "python", "sql", "excel", "power bi", "tableau",
    "aws", "azure", "gcp", "java", "javascript",
    "pandas", "numpy", "spark", "machine learning"
]

In [66]:
text = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["tags"].fillna("")
).str.lower()

In [67]:
def find_skills(text):
    return [
        skill for skill in skills
        if re.search(r"\b" + re.escape(skill) + r"\b", text)
    ]

In [68]:
df["skills"] = text.apply(find_skills)

In [69]:
df["skills"].head()

0    []
1    []
2    []
3    []
4    []
Name: skills, dtype: object

In [70]:
df["skill_count"] = df["skills"].apply(len)

In [71]:
df["skill_count"].describe()

count    2873.000000
mean        0.596589
std         1.095294
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         7.000000
Name: skill_count, dtype: float64

In [72]:
# Convert Skills to Text
df["skills"] = df["skills"].apply(
    lambda x: " | ".join(x)
)

In [73]:
df[["title", "skills"]].head(10)

,title,skills
0,Software Engineer,
1,Senior Product Manager,
2,Working Student Launch Operations (m/f/d),
3,Senior Mechanical Engineer (f/m/d),
4,MES Process and System Engineer (m/f/d),
5,Executive Assistant to the Chief of Staff (m/f/d),
6,Director Value Stream (m/f/d),
7,Cybersecurity Engineer (m/f/d),python
8,Chief Quality Inspector (m/f/d),
9,Strategic Account Executive (DACH),


In [74]:
skills = [
    "python", "sql", "excel", "power bi", "tableau",
    "aws", "azure", "gcp", "java", "javascript",
    "pandas", "numpy", "spark", "machine learning"
]

In [75]:
print("Remote:")
print(df["remote_status"].value_counts())

print("\nJob Category:")
print(df["job_category"].value_counts())

print("\nExperience:")
print(df["experience_level"].value_counts())

print("\nTop Cities:")
print(df["city"].value_counts().head(10))

Remote:
remote_status
On-site    2662
Remote      211
Name: count, dtype: int64

Job Category:
job_category
Other                1678
Software & IT         635
Sales & Marketing     279
Data & Analytics      175
HR                     58
Finance                48
Name: count, dtype: int64

Experience:
experience_level
Mid Level / Other    1578
Senior                683
Lead / Manager        492
Entry Level           120
Name: count, dtype: int64

Top Cities:
city
Berlin     584
Munich     265
Hamburg    167
München    120
London      72
Unknown     67
Paris       53
Köln        50
Cologne     48
Remote      47
Name: count, dtype: int64


In [76]:
df[["title", "skills", "skill_count"]].head(10)

,title,skills,skill_count
0,Software Engineer,,0
1,Senior Product Manager,,0
2,Working Student Launch Operations (m/f/d),,0
3,Senior Mechanical Engineer (f/m/d),,0
4,MES Process and System Engineer (m/f/d),,0
5,Executive Assistant to the Chief of Staff (m/f/d),,0
6,Director Value Stream (m/f/d),,0
7,Cybersecurity Engineer (m/f/d),python,1
8,Chief Quality Inspector (m/f/d),,0
9,Strategic Account Executive (DACH),,0


In [77]:
skill_counts = (
    df["skills"]
    .str.split(" | ")
    .explode()
    .value_counts()
)

skill_counts.head(10)

skills
            1945
|            786
python       398
excel        262
sql          205
aws          195
azure        146
java         130
machine      114
learning     114
Name: count, dtype: int64

In [78]:
skill_counts = skill_counts[skill_counts.index != ""]

In [79]:
skill_counts.head(10)

skills
|             786
python        398
excel         262
sql           205
aws           195
azure         146
java          130
machine       114
learning      114
javascript     86
Name: count, dtype: int64

In [80]:
df.shape

(2873, 19)

In [81]:
df.head()

,slug,company_name,title,description,remote,url,tags,job_types,location,created_at,posted_date,posted_year,posted_month,remote_status,job_category,experience_level,city,skills,skill_count
0,software-engineer-price-comparison-portal-berl...,Preiswecker,Software Engineer,Preiswecker is a price comparison portal in Ge...,False,https://www.preiswecker.com/,Software Engineering | E-Commerce,Full-time,Berlin,2026-08-12 06:40:00,2026-08-12,2026,August,On-site,Software & IT,Mid Level / Other,Berlin,,0
1,senior-product-manager-356253,Verve,Senior Product Manager,Who We Are Verve has created a more efficient ...,False,https://www.arbeitnow.com/jobs/companies/verve...,Product Management,NaN,Unknown,2026-10-01 05:55:15,2026-10-01,2026,October,On-site,Other,Senior,Unknown,,0
2,working-student-launch-operations-parsdorf-bav...,isaraerospace,Working Student Launch Operations (m/f/d),Mission Brief As Working Student Launch Operat...,False,https://www.arbeitnow.com/jobs/companies/isara...,Mission & Launch Operations,NaN,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October,On-site,Other,Mid Level / Other,Parsdorf,,0
3,senior-mechanical-engineer-parsdorf-bavaria-38...,isaraerospace,Senior Mechanical Engineer (f/m/d),Mission Brief We deliver the half of the rocke...,False,https://www.arbeitnow.com/jobs/companies/isara...,Ground Systems,NaN,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October,On-site,Software & IT,Senior,Parsdorf,,0
4,mes-process-and-system-engineer-parsdorf-bavar...,isaraerospace,MES Process and System Engineer (m/f/d),Mission Brief&nbsp; Help build the digital man...,False,https://www.arbeitnow.com/jobs/companies/isara...,Production | Operations & Quality,NaN,"Parsdorf, Bavaria",2026-10-01 05:55:11,2026-10-01,2026,October,On-site,Software & IT,Mid Level / Other,Parsdorf,,0


In [82]:
skill_counts = (
    df["skills"]
    .str.split(r"\s*\|\s*")
    .explode()
    .str.strip()
    .replace("", np.nan)
    .dropna()
    .value_counts()
)

skill_counts.head(15)

skills
python              398
excel               262
sql                 205
aws                 195
azure               146
java                130
machine learning    114
javascript           86
gcp                  60
spark                48
power bi             33
tableau              24
pandas                7
numpy                 6
Name: count, dtype: int64

In [83]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Companies:", df["company_name"].nunique())
print("Cities:", df["city"].nunique())
print("Remote jobs:", df["remote_status"].value_counts().to_dict())
print("Top skills:")
print(skill_counts.head(10))

Rows: 2873
Columns: 19
Companies: 1271
Cities: 536
Remote jobs: {'On-site': 2662, 'Remote': 211}
Top skills:
skills
python              398
excel               262
sql                 205
aws                 195
azure               146
java                130
machine learning    114
javascript           86
gcp                  60
spark                48
Name: count, dtype: int64


In [84]:
df.to_csv("../dataset/job_market_analysis.csv", index=False)